<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/etapa-02-grafos/17%20-%20Problemas%20Hamiltonianos%20e%20Roteamento%20de%20AGV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 17: Problemas Hamiltonianos e Roteamento Logístico de AGVs (TSP)

## 1. Fundamentos Matemáticos: Ciclos Hamiltonianos e o Caixeiro-Viajante (TSP)

No sistema **SCADA-Core / Visão-AGV**, frequentemente o veículo autônomo atua como um coletor ou distribuidor, necessitando visitar um conjunto de estações de trabalho para abastecimento ou recolhimento de peças.

Diferente da inspeção de infraestrutura (Euleriano, focado nas arestas), o objetivo aqui é visitar **todos os vértices (estações)** exatamente uma única vez e retornar à base, minimizando a distância total percorrida. Este é um **Ciclo Hamiltoniano**, e encontrar a rota de menor custo é o clássico **Problema do Caixeiro-Viajante (TSP - Traveling Salesperson Problem)**.

Como o TSP é um problema *NP-Difícil*, o cálculo de força bruta torna-se inviável em tempo real para muitas estações. A abordagem industrial adota duas etapas:
1. **Heurística Construtiva (Nearest Neighbor):** Constrói a rota iterativamente, escolhendo sempre a estação não visitada mais próxima.
2. **Refinamento de Busca Local (2-Opt):** Analisa a rota gerada para remover cruzamentos ineficientes (inversão de segmentos), aproximando a solução do ótimo global.

In [3]:
import math
from typing import Dict, List, Tuple

class OtimizadorTSP_AGV:
    """
    Roteador de Coleta/Distribuição Múltipla para AGVs.
    Resolve o TSP utilizando a heurística do Vizinho Mais Próximo combinada com Refinamento 2-Opt.
    """
    def __init__(self):
        self.estacoes: Dict[str, Tuple[float, float]] = {}

    def adicionar_estacao(self, tag: str, x: float, y: float):
        """Cadastra a coordenada espacial (X, Y) da estação no ambiente do galpão."""
        self.estacoes[tag] = (x, y)

    def _distancia(self, n1: str, n2: str) -> float:
        """Calcula a distância euclidiana reta entre duas estações."""
        x1, y1 = self.estacoes[n1]
        x2, y2 = self.estacoes[n2]
        return math.dist((x1, y1), (x2, y2))

    def custo_total(self, rota: List[str]) -> float:
        """Calcula o custo acumulado (distância total) de uma rota."""
        return sum(self._distancia(rota[i], rota[i+1]) for i in range(len(rota)-1))

    def heuristica_nearest_neighbor(self, inicio: str) -> List[str]:
        """Gera uma rota inicial gulosa escolhendo iterativamente a estação mais próxima."""
        nao_visitados = list(self.estacoes.keys())
        nao_visitados.remove(inicio)
        rota = [inicio]

        atual = inicio
        while nao_visitados:
            # Encontra o vizinho mais próximo entre os não visitados
            proximo = min(nao_visitados, key=lambda no: self._distancia(atual, no))
            rota.append(proximo)
            nao_visitados.remove(proximo)
            atual = proximo

        rota.append(inicio) # Fecha o ciclo retornando à base
        return rota

    def otimizar_2opt(self, rota_inicial: List[str]) -> List[str]:
        """
        Refina a rota removendo cruzamentos ineficientes.
        Avalia a inversão de segmentos de rota para encontrar um custo total menor.
        """
        melhor_rota = rota_inicial
        melhor_custo = self.custo_total(rota_inicial)
        melhoria = True

        while melhoria:
            melhoria = False
            # O primeiro e o último nó são a Base e devem permanecer fixos
            for i in range(1, len(melhor_rota) - 2):
                for j in range(i + 1, len(melhor_rota) - 1):
                    # Inverte o trecho da rota entre os índices i e j
                    nova_rota = melhor_rota[:i] + melhor_rota[i:j+1][::-1] + melhor_rota[j+1:]
                    novo_custo = self.custo_total(nova_rota)

                    if novo_custo < melhor_custo:
                        melhor_rota = nova_rota
                        melhor_custo = novo_custo
                        melhoria = True # Continua o loop até alcançar um ótimo local

        return melhor_rota

## 2. Simulação e Avaliação do Otimizador Logístico

Vamos instanciar a malha de estações com coordenadas (X, Y) reais do galpão. Em seguida, calcularemos a rota de coleta do AGV utilizando a abordagem gulosa (*Nearest Neighbor*) e aplicaremos o *2-Opt* para verificar a economia de bateria/distância alcançada pela eliminação de cruzamentos de trajetória.

In [4]:
otimizador = OtimizadorTSP_AGV()

# Mapeamento Espacial do Galpão (Coordenadas X, Y em metros)
otimizador.adicionar_estacao("ST-01", 0.0, 0.0)      # Estação Base
otimizador.adicionar_estacao("DOC-101", 10.0, 20.0)  # Doca 1
otimizador.adicionar_estacao("ALM-201", 30.0, 10.0)  # Almoxarifado
otimizador.adicionar_estacao("AMO-301", 40.0, 40.0)  # Posto de Amostragem
otimizador.adicionar_estacao("R-101", 15.0, 45.0)    # Reator Químico
otimizador.adicionar_estacao("DEP-401", 5.0, 35.0)   # Depósito Final

print("=== OTIMIZAÇÃO DE ROTA DE COLETA MÚLTIPLA (AGV) ===\n")

# Passo 1: Construção da Rota via Nearest Neighbor
rota_nn = otimizador.heuristica_nearest_neighbor("ST-01")
custo_nn = otimizador.custo_total(rota_nn)
print(f"[PASSO 1] Rota Nearest Neighbor (Gulosa):\n {' -> '.join(rota_nn)}")
print(f" Distância Total: {custo_nn:.2f} m\n")

# Passo 2: Refinamento da Rota via 2-Opt
rota_otimizada = otimizador.otimizar_2opt(rota_nn)
custo_otimizado = otimizador.custo_total(rota_otimizada)
economia = custo_nn - custo_otimizado

print(f"[PASSO 2] Rota Pós-Refinamento (2-Opt):\n {' -> '.join(rota_otimizada)}")
print(f" Distância Otimizada: {custo_otimizado:.2f} m\n")

# Resultado Final
if economia > 0:
    print(f"[SUCESSO SCADA] O algoritmo 2-Opt removeu cruzamentos ineficientes!")
    print(f"Economia gerada: {economia:.2f} metros por ciclo logístico.")
else:
    print("[INFO] A rota inicial Nearest Neighbor já não continha cruzamentos (ótimo local atingido).")

=== OTIMIZAÇÃO DE ROTA DE COLETA MÚLTIPLA (AGV) ===

[PASSO 1] Rota Nearest Neighbor (Gulosa):
 ST-01 -> DOC-101 -> DEP-401 -> R-101 -> AMO-301 -> ALM-201 -> ST-01
 Distância Total: 141.05 m

[PASSO 2] Rota Pós-Refinamento (2-Opt):
 ST-01 -> DOC-101 -> DEP-401 -> R-101 -> AMO-301 -> ALM-201 -> ST-01
 Distância Otimizada: 141.05 m

[INFO] A rota inicial Nearest Neighbor já não continha cruzamentos (ótimo local atingido).
